In [33]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib as mpl

import scipy
import numpy as np
import ipywidgets as widgets

import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

In [34]:
SAMPLERATE = 250000
NUM_CHANNELS = 8
BYTES_PER_SAMPLE = 2 #int16

COLOR_MAP_FOR_TRAJ = {0:cm.Blues, 1:cm.Reds, 2:cm.Greens}
MIC_MARKER_THICKNESS = 2
POINT_SIZE = 75

In [107]:
GRID_SIZE = 25
CIRCLE_RADIUS = 10
SOUND_SPEED_AIR = 343
TIME_DURATION = 0.2
RCVR_COLORS = ['red', 'limegreen', 'blue', 'mediumpurple', 'slategray', 'brown', 'green', 'darkred']
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=RCVR_COLORS)
COLOR_CYCLE = plt.rcParams['axes.prop_cycle'].by_key()['color']
BAT_INIT_DIST = 10

FS = 250000
ASSUMED_BAT_SPEED = 4*np.sqrt(3)/3 # m/s
ASSUMED_BAT_IPI = 0.1 # secs (100ms)
TIMESTEPS = np.arange(0, ((2*BAT_INIT_DIST)/ASSUMED_BAT_SPEED) + ASSUMED_BAT_IPI, ASSUMED_BAT_IPI)

In [108]:
ZMAG_REF_TO_6 = 29.125
ZMAG_REF_TO_7 = 31.6875
XMAG_REF_TO_1 = 17.7
YMAG_REF_TO_6 = 21
YMAG_REF_TO_1 = 37.5
UBNA_ARRAY_MIC_LOCS =  (254/10000) * np.array([[-XMAG_REF_TO_1, -YMAG_REF_TO_1, ZMAG_REF_TO_7],
                    [-XMAG_REF_TO_1, YMAG_REF_TO_1, ZMAG_REF_TO_7],
                    [-XMAG_REF_TO_1, -YMAG_REF_TO_1, 0],
                    [-XMAG_REF_TO_1, YMAG_REF_TO_1, 0],
                    [-XMAG_REF_TO_1, -YMAG_REF_TO_6, -ZMAG_REF_TO_6],
                    [-XMAG_REF_TO_1, YMAG_REF_TO_6, -ZMAG_REF_TO_6],
                    [0, 0, ZMAG_REF_TO_7],
                    [0, 0, 0]])
UBNA_ARRAY_MIC_LOCS

array([[-0.44958  , -0.9525   ,  0.8048625],
       [-0.44958  ,  0.9525   ,  0.8048625],
       [-0.44958  , -0.9525   ,  0.       ],
       [-0.44958  ,  0.9525   ,  0.       ],
       [-0.44958  , -0.5334   , -0.739775 ],
       [-0.44958  ,  0.5334   , -0.739775 ],
       [ 0.       ,  0.       ,  0.8048625],
       [ 0.       ,  0.       ,  0.       ]])

In [109]:
%matplotlib inline

SELECTED_CHANNEL_FOR_REF = 7
MICROPHONES_USED = np.array([1,2,3,4,5,6,7,8])
IND_OF_SELECTED_CHANNEL = np.where(MICROPHONES_USED==(SELECTED_CHANNEL_FOR_REF+1))[0]
SELECTED_MIC_FOR_REF = MICROPHONES_USED[IND_OF_SELECTED_CHANNEL]
NON_REF_MICROPHONES_USED = MICROPHONES_USED[MICROPHONES_USED!=SELECTED_MIC_FOR_REF]
NUM_GOOD_CHANNELS = MICROPHONES_USED.shape[0]
NUM_NONREFCHANNELS = NUM_GOOD_CHANNELS-1

In [110]:
def simulate_N_receivers(time_step, spacing):
    N = UBNA_ARRAY_MIC_LOCS.shape[0]
    rN_x = UBNA_ARRAY_MIC_LOCS[:,0] * spacing
    rN_y = UBNA_ARRAY_MIC_LOCS[:,1] * spacing
    rN_z = UBNA_ARRAY_MIC_LOCS[:,2] * spacing
    A_LOCS_MAT = UBNA_ARRAY_MIC_LOCS
    A_locs_mat_wrt_ref_channel = A_LOCS_MAT
    A_locs_mat_tdoa_meters = A_locs_mat_wrt_ref_channel[(NON_REF_MICROPHONES_USED-1)]

    x0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][0]
    y0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][1]
    z0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][2]
    xm = A_locs_mat_tdoa_meters[:,0].reshape((NUM_NONREFCHANNELS, 1))
    ym = A_locs_mat_tdoa_meters[:,1].reshape((NUM_NONREFCHANNELS, 1))
    zm = A_locs_mat_tdoa_meters[:,2].reshape((NUM_NONREFCHANNELS, 1))
    
    trace_offsets = (60/N) * np.linspace(-0.1, 0.1, N)

    time_transmitted = TIME_DURATION/2
    t = np.linspace(0, TIME_DURATION, int(FS*TIME_DURATION))
    signal = scipy.signal.unit_impulse(t.size, idx=int(time_transmitted*FS)) * 1/N
    signal_duration = TIME_DURATION/2

    time_steps = np.arange(0, time_step+ASSUMED_BAT_IPI, ASSUMED_BAT_IPI)
    time_idx = np.where(np.isclose(time_steps, time_step, atol=1e-2))[0][0]
    x_arr = ASSUMED_BAT_SPEED * (time_steps) - (BAT_INIT_DIST)
    y_arr = ASSUMED_BAT_SPEED * (time_steps) - (BAT_INIT_DIST)
    z_arr = (-ASSUMED_BAT_SPEED * time_steps) + (2*BAT_INIT_DIST)
    pos_x = x_arr[time_idx]
    pos_y = y_arr[time_idx]
    pos_z = z_arr[time_idx]

    dist_from_source_to_r_n_arr = np.sqrt((x_arr[None, :] - rN_x[:, None]) ** 2 + (y_arr[None, :] - rN_y[:, None]) ** 2 + (z_arr[None, :] - rN_z[:, None]) ** 2)
    time_to_rn_array = dist_from_source_to_r_n_arr / SOUND_SPEED_AIR
    receive_signaln_for_timestep = signal[:, None, None]
    signal_duration_samples = np.arange(int(signal_duration * FS))
    rn_indices = ((time_transmitted - time_to_rn_array) * FS).astype(int) + signal_duration_samples[:, None, None]
    chunk_of_received_signaln_in_window_timestep = np.take_along_axis(receive_signaln_for_timestep, rn_indices, axis=0)

    fig = plt.figure(figsize=(10, 5))
    plt.rcParams.update({'font.size':12})
    gs = gridspec.GridSpec(1, 2, width_ratios=[4, 4], wspace=0.3) 
    marker_handles = [Line2D([0], [0], marker='o', color='w', label=f'rcvr #{i+1}', markerfacecolor=COLOR_CYCLE[i], markersize=6) for i in range(N)]

    map_ax_3d = plt.subplot(gs[0], projection='3d')
    map_ax_3d.scatter(pos_x, pos_y, pos_z, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    received_ax = plt.subplot(gs[0, 1])
    map_ax_3d.scatter(rN_x, rN_y, rN_z, s=10, c=COLOR_CYCLE[:N])
    map_ax_3d.plot(x_arr, y_arr, z_arr, color='k', alpha=0.5, zorder=4)

    received_ax.plot(1000*(np.tile(signal_duration_samples / FS, (N, 1)).T), chunk_of_received_signaln_in_window_timestep[:,:,time_idx]+trace_offsets)
    map_ax_3d.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_ylim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_zlim(-GRID_SIZE / 2, GRID_SIZE)
    map_ax_3d.set_xlabel("X (meters)")
    map_ax_3d.set_ylabel("Y (meters)")
    map_ax_3d.set_zlabel("Z (meters)")
    map_ax_3d.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")
    map_ax_3d.view_init(elev=10, azim=10)

    received_ax.set_xlabel("Time (ms)")
    received_ax.set_ylabel("Amplitude")
    received_ax.set_title("Received voltage signals (t=k)")
    received_ax.grid(which='both')
    received_ax.set_yticks([], [])
    received_ax.set_xlim(0, 1000*signal_duration)
    received_ax.set_ylim(-1, 1)
    received_ax.legend(handles=marker_handles, ncol=2, bbox_to_anchor=(-0.25, -0.25), loc='lower right')
    plt.show()


time_slider = widgets.FloatSlider(
    value=TIMESTEPS[-1], min=0, max=TIMESTEPS[-1], step=ASSUMED_BAT_IPI, description="Time (k)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

N_slider = widgets.IntSlider(
    value=0, min=1, max=5, step=1, description="# receivers", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="300px")
)

spacing_slider = widgets.FloatSlider(
    value=1, min=0, max=5, step=0.01, description="Spacing (m)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

interactive_plot = widgets.interactive(simulate_N_receivers, time_step=time_slider, N=N_slider, spacing=spacing_slider)
display(interactive_plot)

interactive(children=(FloatSlider(value=8.700000000000001, description='Time (k)', layout=Layout(width='1000px…

In [156]:
def simulate_N_receivers(time_step, spacing):
    N = UBNA_ARRAY_MIC_LOCS.shape[0]
    rN_x = UBNA_ARRAY_MIC_LOCS[:,0] * spacing
    rN_y = UBNA_ARRAY_MIC_LOCS[:,1] * spacing
    rN_z = UBNA_ARRAY_MIC_LOCS[:,2] * spacing
    A_LOCS_MAT = UBNA_ARRAY_MIC_LOCS
    A_locs_mat_wrt_ref_channel = A_LOCS_MAT
    A_locs_mat_tdoa_meters = A_locs_mat_wrt_ref_channel[(NON_REF_MICROPHONES_USED-1)]

    x0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][0]
    y0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][1]
    z0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][2]
    xm = A_locs_mat_tdoa_meters[:,0].reshape((NUM_NONREFCHANNELS, 1))
    ym = A_locs_mat_tdoa_meters[:,1].reshape((NUM_NONREFCHANNELS, 1))
    zm = A_locs_mat_tdoa_meters[:,2].reshape((NUM_NONREFCHANNELS, 1))
    
    trace_offsets = (60/N) * np.linspace(-0.1, 0.1, N)

    time_transmitted = TIME_DURATION/2
    t = np.linspace(0, TIME_DURATION, int(FS*TIME_DURATION))
    signal = scipy.signal.unit_impulse(t.size, idx=int(time_transmitted*FS)) * 1/N
    signal_duration = TIME_DURATION/2

    time_steps = np.arange(0, time_step+ASSUMED_BAT_IPI, ASSUMED_BAT_IPI)
    time_idx = np.where(np.isclose(time_steps, time_step, atol=1e-2))[0][0]
    x_arr = ASSUMED_BAT_SPEED * (time_steps) - (BAT_INIT_DIST)
    y_arr = ASSUMED_BAT_SPEED * (time_steps) - (BAT_INIT_DIST)
    z_arr = (-ASSUMED_BAT_SPEED * time_steps) + (2*BAT_INIT_DIST)
    pos_x = x_arr[time_idx]
    pos_y = y_arr[time_idx]
    pos_z = z_arr[time_idx]

    dist_from_source_to_r_n_arr = np.sqrt((x_arr[None, :] - rN_x[:, None]) ** 2 + (y_arr[None, :] - rN_y[:, None]) ** 2 + (z_arr[None, :] - rN_z[:, None]) ** 2)
    time_to_rn_array = dist_from_source_to_r_n_arr / SOUND_SPEED_AIR
    receive_signaln_for_timestep = signal[:, None, None]
    signal_duration_samples = np.arange(int(signal_duration * FS))
    rn_indices = ((time_transmitted - time_to_rn_array) * FS).astype(int) + signal_duration_samples[:, None, None]
    chunk_of_received_signaln_in_window_timestep = np.take_along_axis(receive_signaln_for_timestep, rn_indices, axis=0)

    true_time_delay_mics_to_selected_ref_channel = ((dist_from_source_to_r_n_arr[:,:] / SOUND_SPEED_AIR) - (dist_from_source_to_r_n_arr[SELECTED_CHANNEL_FOR_REF,:] / SOUND_SPEED_AIR)).T
    true_d_mics_to_ref = np.delete(true_time_delay_mics_to_selected_ref_channel * SOUND_SPEED_AIR, IND_OF_SELECTED_CHANNEL, axis=1)

    fig = plt.figure(figsize=(10, 5))
    plt.rcParams.update({'font.size':12})
    gs = gridspec.GridSpec(1, 2, width_ratios=[4, 4], wspace=0.3) 
    marker_handles = [Line2D([0], [0], marker='o', color='w', label=f'rcvr #{i+1}', markerfacecolor=COLOR_CYCLE[i], markersize=6) for i in range(N)]

    map_ax_3d = plt.subplot(gs[0], projection='3d')
    map_ax_3d.scatter(pos_x, pos_y, pos_z, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    received_ax = plt.subplot(gs[0, 1])
    map_ax_3d.scatter(rN_x, rN_y, rN_z, s=10, c=COLOR_CYCLE[:N])
    map_ax_3d.plot(x_arr, y_arr, z_arr, color='k', alpha=0.5, zorder=4)

    norm = plt.Normalize(vmin=0, vmax=TIMESTEPS.shape[0])
    source_locs = np.zeros((4, true_d_mics_to_ref.shape[0]-1), dtype=np.float64)
    cond_a_vals = np.zeros(true_d_mics_to_ref.shape[0]-1, dtype=np.float64)
    for i in range(true_d_mics_to_ref.shape[0]-1):
        dm0_column = true_d_mics_to_ref[i,:].reshape((NUM_NONREFCHANNELS, 1))
        A_mat = np.hstack(((A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF] - A_locs_mat_tdoa_meters), dm0_column))

        wm0 = ((dm0_column**2) - (xm**2) + (x0**2) - (ym**2) + (y0**2) - (zm**2) + (z0**2))/2

        xs, residuals, rank, sing_vals = scipy.linalg.lstsq(A_mat, wm0)
        source_locs[:,i] = xs.reshape((1,4))
        cond_a_vals[i] = np.linalg.cond(A_mat)
        
        line_color = COLOR_MAP_FOR_TRAJ[1](norm(i))
        map_ax_3d.scatter(source_locs[0,i], source_locs[1,i], source_locs[2,i], color=line_color, edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)

    received_ax.plot(1000*(np.tile(signal_duration_samples / FS, (N, 1)).T), chunk_of_received_signaln_in_window_timestep[:,:,time_idx]+trace_offsets)
    map_ax_3d.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_ylim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_zlim(-GRID_SIZE / 2, GRID_SIZE)
    map_ax_3d.set_xlabel("X (meters)")
    map_ax_3d.set_ylabel("Y (meters)")
    map_ax_3d.set_zlabel("Z (meters)")
    map_ax_3d.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")
    map_ax_3d.view_init(elev=10, azim=10)

    received_ax.set_xlabel("Time (ms)")
    received_ax.set_ylabel("Amplitude")
    received_ax.set_title("Received voltage signals (t=k)")
    received_ax.grid(which='both')
    received_ax.set_yticks([], [])
    received_ax.set_xlim(0, 1000*signal_duration)
    received_ax.set_ylim(-1, 1)
    received_ax.legend(handles=marker_handles, ncol=2, bbox_to_anchor=(-0.25, -0.25), loc='lower right')
    
    plt.figure(figsize=(10, 2))
    plt.plot(cond_a_vals, marker='.')
    plt.grid(which='both')
    plt.ylabel('Cond(A)')
    plt.xlabel('Source location time step')
    plt.ylim(-50, 300)

    plt.show()


time_slider = widgets.FloatSlider(
    value=TIMESTEPS[-1], min=0, max=TIMESTEPS[-1], step=ASSUMED_BAT_IPI, description="Time (k)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

N_slider = widgets.IntSlider(
    value=0, min=1, max=5, step=1, description="# receivers", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="300px")
)

spacing_slider = widgets.FloatSlider(
    value=1, min=0, max=5, step=0.01, description="Spacing (m)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

interactive_plot = widgets.interactive(simulate_N_receivers, time_step=time_slider, N=N_slider, spacing=spacing_slider)
display(interactive_plot)

interactive(children=(FloatSlider(value=8.700000000000001, description='Time (k)', layout=Layout(width='1000px…

In [154]:
def simulate_N_receivers(time_step, spacing):
    N = UBNA_ARRAY_MIC_LOCS.shape[0]
    rN_x = UBNA_ARRAY_MIC_LOCS[:,0] * spacing
    rN_y = UBNA_ARRAY_MIC_LOCS[:,1] * spacing
    rN_z = UBNA_ARRAY_MIC_LOCS[:,2] * spacing
    A_LOCS_MAT = UBNA_ARRAY_MIC_LOCS
    A_locs_mat_wrt_ref_channel = A_LOCS_MAT
    A_locs_mat_tdoa_meters = A_locs_mat_wrt_ref_channel[(NON_REF_MICROPHONES_USED-1)]

    x0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][0]
    y0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][1]
    z0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][2]
    xm = A_locs_mat_tdoa_meters[:,0].reshape((NUM_NONREFCHANNELS, 1))
    ym = A_locs_mat_tdoa_meters[:,1].reshape((NUM_NONREFCHANNELS, 1))
    zm = A_locs_mat_tdoa_meters[:,2].reshape((NUM_NONREFCHANNELS, 1))
    
    trace_offsets = (60/N) * np.linspace(-0.1, 0.1, N)

    time_transmitted = TIME_DURATION/2
    t = np.linspace(0, TIME_DURATION, int(FS*TIME_DURATION))
    signal = scipy.signal.unit_impulse(t.size, idx=int(time_transmitted*FS)) * 1/N
    signal_duration = TIME_DURATION/2

    time_steps = np.arange(0, time_step+ASSUMED_BAT_IPI, ASSUMED_BAT_IPI)
    time_idx = np.where(np.isclose(time_steps, time_step, atol=1e-2))[0][0]
    x_arr = ASSUMED_BAT_SPEED * (time_steps) - (BAT_INIT_DIST)
    y_arr = ASSUMED_BAT_SPEED * (time_steps) - (BAT_INIT_DIST)
    z_arr = (-ASSUMED_BAT_SPEED * time_steps) + (2*BAT_INIT_DIST)
    pos_x = x_arr[time_idx]
    pos_y = y_arr[time_idx]
    pos_z = z_arr[time_idx]

    dist_from_source_to_r_n_arr = np.sqrt((x_arr[None, :] - rN_x[:, None]) ** 2 + (y_arr[None, :] - rN_y[:, None]) ** 2 + (z_arr[None, :] - rN_z[:, None]) ** 2)
    time_to_rn_array = dist_from_source_to_r_n_arr / SOUND_SPEED_AIR
    receive_signaln_for_timestep = signal[:, None, None]
    signal_duration_samples = np.arange(int(signal_duration * FS))
    rn_indices = ((time_transmitted - time_to_rn_array) * FS).astype(int) + signal_duration_samples[:, None, None]
    chunk_of_received_signaln_in_window_timestep = np.take_along_axis(receive_signaln_for_timestep, rn_indices, axis=0)

    sample_received_at_rn = np.argmax(chunk_of_received_signaln_in_window_timestep, axis=0)
    time_received_at_rn = (sample_received_at_rn / FS).T
    t_delay_ref_to_selected_channel = time_received_at_rn[:,IND_OF_SELECTED_CHANNEL]
    time_delay_mics_to_selected_ref_channel = (time_received_at_rn[:]) - (t_delay_ref_to_selected_channel.reshape((len(t_delay_ref_to_selected_channel), 1)))
    time_delay_mics_to_selected_ref_channel_no_ref = np.delete(time_delay_mics_to_selected_ref_channel, IND_OF_SELECTED_CHANNEL, axis=1)
    measured_d_mics_to_ref = (time_delay_mics_to_selected_ref_channel_no_ref * SOUND_SPEED_AIR)

    fig = plt.figure(figsize=(10, 5))
    plt.rcParams.update({'font.size':12})
    gs = gridspec.GridSpec(1, 2, width_ratios=[4, 4], wspace=0.3) 
    marker_handles = [Line2D([0], [0], marker='o', color='w', label=f'rcvr #{i+1}', markerfacecolor=COLOR_CYCLE[i], markersize=6) for i in range(N)]

    map_ax_3d = plt.subplot(gs[0], projection='3d')
    map_ax_3d.scatter(pos_x, pos_y, pos_z, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    received_ax = plt.subplot(gs[0, 1])
    map_ax_3d.scatter(rN_x, rN_y, rN_z, s=10, c=COLOR_CYCLE[:N])

    norm = plt.Normalize(vmin=0, vmax=TIMESTEPS.shape[0])
    source_locs = np.zeros((4, measured_d_mics_to_ref.shape[0]-1), dtype=np.float64)
    cond_a_vals = np.zeros(measured_d_mics_to_ref.shape[0]-1, dtype=np.float64)
    for i in range(measured_d_mics_to_ref.shape[0]-1):
        dm0_column = measured_d_mics_to_ref[i,:].reshape((NUM_NONREFCHANNELS, 1))
        A_mat = np.hstack(((A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF] - A_locs_mat_tdoa_meters), dm0_column))

        wm0 = ((dm0_column**2) - (xm**2) + (x0**2) - (ym**2) + (y0**2) - (zm**2) + (z0**2))/2

        xs, residuals, rank, sing_vals = scipy.linalg.lstsq(A_mat, wm0)
        source_locs[:,i] = xs.reshape((1,4))
        cond_a_vals[i] = np.linalg.cond(A_mat)
        
        line_color = COLOR_MAP_FOR_TRAJ[1](norm(i))
        map_ax_3d.scatter(source_locs[0,i], source_locs[1,i], source_locs[2,i], color=line_color, edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)

    received_ax.plot(1000*(np.tile(signal_duration_samples / FS, (N, 1)).T), chunk_of_received_signaln_in_window_timestep[:,:,time_idx]+trace_offsets)
    map_ax_3d.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_ylim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_zlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_3d.set_xlabel("X (meters)")
    map_ax_3d.set_ylabel("Y (meters)")
    map_ax_3d.set_zlabel("Z (meters)")
    map_ax_3d.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")
    map_ax_3d.legend(handles=marker_handles, ncol=2, loc='upper right')

    received_ax.scatter(1e3*(sample_received_at_rn[:,time_idx]/FS), 
                        np.diagonal((chunk_of_received_signaln_in_window_timestep[:,:,time_idx])[sample_received_at_rn[:,time_idx],:]) + trace_offsets,
                        s=50, marker='*', zorder=2)
    received_ax.set_xlabel("Time (ms)")
    received_ax.set_ylabel("Amplitude")
    received_ax.set_title("Received voltage signals")
    received_ax.grid(which='both')
    received_ax.set_yticks([], [])
    received_ax.set_xlim(0, 1000*signal_duration)
    received_ax.set_ylim(-1, 1)
    
    plt.figure(figsize=(10, 2))
    plt.plot(cond_a_vals, marker='.')
    plt.grid(which='both')
    plt.ylabel('Cond(A)')
    plt.xlabel('Source location time step')
    plt.ylim(-50, 300)

    plt.show()


time_slider = widgets.FloatSlider(
    value=TIMESTEPS[-1], min=0, max=TIMESTEPS[-1], step=ASSUMED_BAT_IPI, description="Time (k)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

N_slider = widgets.IntSlider(
    value=0, min=1, max=5, step=1, description="# receivers", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="300px")
)

spacing_slider = widgets.FloatSlider(
    value=1, min=0, max=5, step=0.01, description="Spacing (m)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

interactive_plot = widgets.interactive(simulate_N_receivers, time_step=time_slider, N=N_slider, spacing=spacing_slider)
display(interactive_plot)

interactive(children=(FloatSlider(value=8.700000000000001, description='Time (k)', layout=Layout(width='1000px…